In [1]:
import os 
import json 
from dotenv import load_dotenv
from openai import OpenAI
import pandas as pd 
from IPython.display import Markdown, display, update_display

In [2]:
load_dotenv (override= True) 
api_key = os.getenv ('OPENAI_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
MODEL = 'gpt-5-nano'
openai = OpenAI()

API key looks good so far


In [3]:
def Inventory_to_JSON (file) : 
  df= pd.read_csv (file)
  data_json = df.head(26)
  return data_json

In [4]:
DataFrame = Inventory_to_JSON ("Inventory_Dataset.csv")

In [5]:
json_system_prompt = """
You are a data analyst specialized in structured inventory processing. 
Your sole task is to convert raw inventory data into a clean JSON object. 

RULES FOR CLASSIFICATION:
1. Critical: (Stock_Quantity < Reorder_Level AND Sales_Volume > 80) OR (Expiration_Date within 30 days).
2. Warning: Stock_Quantity < Reorder_Level (but Sales_Volume <= 80).

OUTPUT REQUIREMENTS:
- Output ONLY a valid JSON object.
- The JSON must contain a single key named "alerts" which holds the array of identified products.
- DO NOT include any introductory text, apologies, explanations, or metadata.
- If no items meet the criteria, return {"alerts": []}.

Example Output Format:
{
  "alerts": [
    {"product": "Bell Pepper", "status": "Critical", "reason": "Low Stock & High Sales", "action": "Reorder Immediately"},
    {"product": "Milk", "status": "Warning", "reason": "Near Expiry", "action": "Clearance Sale"}
  ]
}

"""

In [6]:
def get_user_prompt (data) : 
  user_prompt = f""" 
Look at ALL products in the dataset below.
    For EACH product that meets the 'Critical' or 'Warning' criteria, create a JSON object.
    Return the result as a LIST of JSON objects [{{...}}, {{...}}].
    there is Data : 
    {data}
  """
  return user_prompt

In [7]:
get_user_prompt (DataFrame)

" \nLook at ALL products in the dataset below.\n    For EACH product that meets the 'Critical' or 'Warning' criteria, create a JSON object.\n    Return the result as a LIST of JSON objects [{...}, {...}].\n    there is Data : \n             Product_Name             Catagory Supplier_Name  \\\n0         Bell Pepper  Fruits & Vegetables        Eimbee   \n1       Vegetable Oil          Oils & Fats      Digitube   \n2     Parmesan Cheese                Dairy        BlogXS   \n3              Carrot  Fruits & Vegetables        Avaveo   \n4              Garlic  Fruits & Vegetables          Katz   \n5               Lemon  Fruits & Vegetables          Yata   \n6       Coconut Sugar      Grains & Pulses          Lazz   \n7           Anchovies              Seafood       Zoonder   \n8              Cheese                Dairy          Oozz   \n9              Yogurt                Dairy     Jaxnation   \n10     Cheddar Cheese                Dairy       Gabcube   \n11        Avocado Oil          Oils

In [8]:
def get_json_data (df) : 
  response = openai.chat.completions.create ( 
    model = MODEL,
    messages = [
      { "role" : "system", "content" : json_system_prompt} ,
      {"role" : "user" , "content" : get_user_prompt(df) } 
     ], 

     response_format= {"type" : "json_object"}
  )

  result = response.choices[0].message.content
  Jonas_Data = json.loads(result)
  return Jonas_Data

In [9]:
Result = get_json_data (DataFrame)

In [10]:
Result

{'alerts': [{'product': 'Bell Pepper',
   'status': 'Critical',
   'reason': 'Low Stock & High Sales',
   'action': 'Reorder Immediately'},
  {'product': 'Vegetable Oil',
   'status': 'Warning',
   'reason': 'Stock below reorder level',
   'action': 'Reorder Soon'},
  {'product': 'Parmesan Cheese',
   'status': 'Warning',
   'reason': 'Stock below reorder level',
   'action': 'Reorder Soon'},
  {'product': 'Carrot',
   'status': 'Warning',
   'reason': 'Stock below reorder level',
   'action': 'Reorder Soon'},
  {'product': 'Coconut Sugar',
   'status': 'Warning',
   'reason': 'Stock below reorder level',
   'action': 'Reorder Soon'},
  {'product': 'Avocado Oil',
   'status': 'Warning',
   'reason': 'Stock below reorder level',
   'action': 'Reorder Soon'},
  {'product': 'Digestive Biscuit',
   'status': 'Warning',
   'reason': 'Stock below reorder level',
   'action': 'Reorder Soon'},
  {'product': 'Cauliflower',
   'status': 'Warning',
   'reason': 'Stock below reorder level',
   'ac

In [11]:
Analysis_system_prompt = """
"You are a professional Business Strategy Consultant and Inventory Analyst.
Your role is to analyze structured JSON inventory data and provide a high-level executive report. 
Your language must be formal, precise, and data-driven. 
Focus on providing actionable insights regarding supply chain risks, sales optimization, and operational efficiency.
 """

In [12]:
def get_analysis_user_prompt (json_data ) :
  user_prompt  = f"""
  The following JSON dataset contains critical inventory alerts:
  {json_data}

Please conduct a professional business analysis based on this data. Your report should include:

An Assessment of current stock risks.

Strategic recommendations for procurement and sales.

A concluding summary on how to optimize inventory turnover.
   """

  return user_prompt

In [13]:
def stream_analysis (json_data) : 
  stream = openai.chat.completions.create (
    model = MODEL,
    messages= [ 
      {"role" : "system" , "content" : Analysis_system_prompt},
      {
        "role" : "user" , "content" : get_analysis_user_prompt(json_data)
      }
    ],
    stream = True
  )
  response = ""
  display_handle = display(Markdown(""), display_id=True)
  for chunk in stream:
    response += chunk.choices[0].delta.content or ''
    update_display(Markdown(response), display_id=display_handle.display_id)

In [14]:
stream_analysis (Result)

Executive Inventory Risk and Strategy Report

Context
The provided alerts identify stock conditions across 12 SKUs. Summary:
- Critical: 2 items (Bell Pepper; Sourdough Bread) due to Low Stock & High Sales.
- Warning: 10 items (Vegetable Oil; Parmesan Cheese; Carrot; Coconut Sugar; Avocado Oil; Digestive Biscuit; Cauliflower; Eggplant; Whole Wheat Flour; Mango) due to Stock below reorder level.

Assessment of current stock risks

1) Immediate stock-out risk and revenue exposure
- Critical items with high sales velocity and depleted stock (Bell Pepper and Sourdough Bread) create a clear risk of missed sales, customer dissatisfaction, and potential substitution to less desirable alternatives.
- While exact on-hand quantities and lead times are not disclosed, the “Low Stock & High Sales” signal for these two items indicates a critical need to accelerate replenishment to preserve service levels.

2) Broad replenishment vulnerability
- Ten items flagged as “Warning” show stock below reorder level across a broad spectrum of categories (produce, ingredients, bakery items, and packaged goods). This pattern suggests either under-replenishment, demand upticks, or potential forecasting gaps.
- If lead times are non-trivial, even modest daily demand could rapidly exhaust available stock, risking stockouts on multiple SKUs in a short window.

3) Perishability and waste risk
- Several items are perishable or semi-perishable (Bell Pepper, Sourdough Bread, Mango, Carrot, Cauliflower, Eggplant). Perishability magnifies risk: stockouts for perishables reduce sales, while overstocking can lead to spoilage.
- For non-perishables (e.g., Vegetable Oil, Parmesan Cheese, Whole Wheat Flour, Coconut Sugar, Digestive Biscuit), stockouts still harm sales but spoilage risk is lower; however, supply disruptions could still cause last-minute shortages.

4) Supply chain and forecasting implications
- The concentration of warnings around a diverse set of SKUs implies potential gaps in forecast accuracy, replenishment cadence, or supplier performance. Without supplier lead times and on-hand data, it is prudent to assume exposure to variability in demand, supplier capacity, and logistics.

5) Operational risk indicators
- The volume of “Below reorder level” alerts across many items signals a possible systemic issue in inventory policy (e.g., insufficient safety stock, infrequent reorders, or delayed replenishment cycles). This warrants an audit of replenishment rules and data integrity (accuracy of on-hand counts, lead times, and demand signals).

Strategic recommendations for procurement and sales

A) Immediate actions (critical items)
- Bell Pepper: Reorder Immediately. Verify preferred SKU variety, origin, and cold-chain constraints. If possible, place expedited or air-freight options to minimize stockout duration. Align delivery with peak sales window to protect service levels.
- Sourdough Bread: Reorder Immediately. Assess shelf life, bakery capacity, and packaging. Consider short-term expedited shipment or secondary suppliers to reduce risk of stockouts during demand surges. Prepare for FIFO-driven merchandising to prevent waste.

B) Short-term actions for warning items (priority sequencing)
For all ten warning SKUs, implement the following within the next 1–2 weeks:
- Validate forecast-to-actual demand: cross-check historical sales velocity and seasonality with current trends. Adjust forecast priors accordingly.
- Reorder now with quantity guidance:
  - Use a demand-driven approach: target replenishment quantities that cover lead times plus a safety stock cushion. A practical starting rule: 1–2 weeks of anticipated usage in safety stock for perishables; 2–4 weeks for non-perishables with longer supplier lead times.
  - Consider tiered replenishment: for items with longer lead times or higher variability (e.g., certain oils, flour, or packaged goods), plan incremental orders to avoid bulk-locking capital.
- Verify supplier readiness: confirm lead times, minimum order quantities (MOQs), capacity for expedited shipments if needed, and alternative supplier viability.
- Perishables-focused controls: enforce tighter FIFO, date-code tracking, and spoilage reporting. Flag near-expiry items for rapid turnover through promotions or bundling.

C) Inventory strategy and policy improvements (medium term)
- Replenishment policy redesign:
  - Implement ABC analysis to categorize SKUs by annual consumption value and demand variability. Allocate more safety stock and more aggressive replenishment for A-items (high value/high turnover), moderate for B-items, and conservative for C-items.
  - Establish explicit service level targets by category (e.g., 95%+ for critical and high-turnover items; 90–95% for mid-pri items; minimum stock coverage for low-turnover items).
- Safety stock and reorder point optimization:
  - Move from static reorder points to demand- and lead-time-driven points. Calculate ROP = (Average daily usage × Lead time) + Safety stock, with safety stock derived from historical variance and desired service level.
  - Introduce regular reviews (daily or every 2–3 days for critical items) to rebalance stock as forecast error or supplier performance changes.
- Supplier risk management:
  - Diversify supplier base for critical items to reduce single-source risk; negotiate secondary suppliers and/or backup capacity.
  - Consider vendor-managed inventory (VMI) or consignment for highly dynamic SKUs where feasible.
- Perishables management:
  - Tighten expiration/date controls, implement stricter FIFO discipline, and use near-expiry promotions to reduce waste.
  - Evaluate alternate packaging or SKUs with longer shelf life to hedge demand volatility.

D) Sales optimization to protect turnover and margins
- Promotions and bundling:
  - Create time-limited promotions or bundles for near-expiry items to accelerate turnover, especially for items in warning status.
  - Use price incentives or value bundles to drive higher basket size and reduce the risk of waste for perishables.
- SKU rationalization and assortment planning:
  - Review whether all warning SKUs are essential. If some items have persistent low turns or high waste rates, consider reducing assortment or substituting with higher-turn alternatives.
- Demand shaping:
  - Coordinate with marketing to align promotions with replenishment cycles, ensuring stock availability during peak demand windows.

E) Operational excellence and visibility
- Inventory control discipline:
  - Implement or strengthen cycle counts and blind counts to improve data accuracy, particularly for items flagged in alerts.
- Real-time analytics and dashboards:
  - Establish a live inventory health dashboard showing stock levels, service levels, days of supply, turnover rates, and expand to include forecast accuracy and lead-time variability.
- Cross-functional governance:
  - Create a monthly S&OP-like cadence with procurement, merchandising, and store/branch operations to review forecasts, stock status, and replenishment plans.

Concluding summary: optimizing inventory turnover

- The current alerts signal a material risk to service levels, with two critical items warranting immediate replenishment and ten additional items requiring rapid attention to prevent cascading stockouts.
- To optimize turnover, adopt an integrated, forecast-driven replenishment framework:
  - Prioritize critical SKUs for aggressive replenishment and supply-chain protection.
  - Implement categorized safety stock and reorder-point rules (ABC analysis) to align stocking levels with value and turnover.
  - Diversify suppliers and consider collaborative inventory arrangements for risk mitigation.
  - Strengthen perishables management to minimize waste and protect gross margins.
  - Use targeted promotions to accelerate turnover for slow-moving or near-expiry items without eroding overall margins.
- Establish a data-driven governance process with real-time visibility, disciplined forecasting, and continuous improvement to both service levels and inventory efficiency.

Next steps suggested
1) Immediate: confirm on-hand quantities, lead times, and order quantities for Bell Pepper and Sourdough Bread; place urgent replenishment orders and explore expedited shipping where viable.
2) Within 1–2 weeks: validate forecasts for all warning SKUs, determine reorder quantities, and initiate replenishment with appropriate safety stock buffers.
3) Within 4–8 weeks: implement ABC-based replenishment policy, establish safety-stock targets, set up an inventory health dashboard, and formalize supplier diversification and risk-mitigation plans.

If you can provide on-hand counts, lead times, and average weekly usage per SKU, I can translate these recommendations into precise reorder quantities, safety stock targets, and a prioritized action plan with concrete milestones and timelines.